# nanoTabPFN Pretraining Notebook

This notebook demonstrates how to pretrain the **nanoTabPFN** model on a synthetic
prior stored in an HDF5 file.

It uses the modular implementation provided in:

- `data_module.py`
- `model_module.py`
- `training_module.py`

## Instructions

- Place the prior data file (e.g. `300k_150x5_2.h5`) in the same directory as this notebook,
  or update the `prior_path` variable below.
- The default hyperparameters match the configuration used in the nanoTabPFN paper.
- Training periodically evaluates on a small real dataset (breast cancer).

⚠️ **Note**:  
The prior file can be very large (hundreds of thousands of synthetic tasks).
This notebook is intended for educational or small-scale experiments.


In [1]:
import numpy as np
import torch

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

from data_module import PriorDumpDataLoader, get_default_device
from model_module import NanoTabPFNModel, NanoTabPFNClassifier
from training_module import train, eval_model, set_randomness_seed


In [2]:
# Reproducibility
set_randomness_seed(47)

# Path to the HDF5 prior dump
prior_path = "300k_150x5_2.h5"

# === Model hyperparameters (exactly as in the paper) ===
embedding_size = 96
num_attention_heads = 4
mlp_hidden_size = 192
num_layers = 3
num_outputs = 2

# === Training hyperparameters ===
num_steps = 2500       # number of batches
batch_size = 32        # tasks per batch
learning_rate = 2e-4

device = get_default_device()
print("Using device:", device)

# Initialize prior dataloader
prior_loader = PriorDumpDataLoader(
    filename=prior_path,
    num_steps=num_steps,
    batch_size=batch_size,
    device=device,
)

# Initialize model
model = NanoTabPFNModel(
    embedding_size=embedding_size,
    num_attention_heads=num_attention_heads,
    mlp_hidden_size=mlp_hidden_size,
    num_layers=num_layers,
    num_outputs=num_outputs,
)

# Small real dataset for evaluation (same as repo example)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=0
)

benchmark_datasets = [
    (X_train, X_test, y_train, y_test)
]


Using device: cuda


In [3]:
trained_model, history = train(
    model=model,
    prior=prior_loader,
    lr=learning_rate,
    device=device,
    steps_per_eval=25,
    eval_func=None,   # no external eval; still prints loss every 25 steps
)


time     4.9s | loss  0.4969
time     6.8s | loss  0.5288
time     8.7s | loss  0.5391
time    10.6s | loss  0.4677
time    12.5s | loss  0.5260
time    14.4s | loss  0.5034
time    16.2s | loss  0.5351
time    18.1s | loss  0.4812
time    20.0s | loss  0.4659
time    21.9s | loss  0.5316
time    23.8s | loss  0.5715
time    25.6s | loss  0.5279
time    27.5s | loss  0.5092
time    29.3s | loss  0.5440
time    31.2s | loss  0.4754
time    33.1s | loss  0.4847
time    35.0s | loss  0.4825
time    36.9s | loss  0.4710
time    38.8s | loss  0.5095
time    40.8s | loss  0.5242
time    42.6s | loss  0.5038
time    44.5s | loss  0.4681
time    46.4s | loss  0.4423
time    48.3s | loss  0.4751
time    50.2s | loss  0.5762
time    52.1s | loss  0.5030
time    54.1s | loss  0.4844
time    56.0s | loss  0.5293
time    57.9s | loss  0.5270
time    59.8s | loss  0.4992
time    61.7s | loss  0.5125
time    63.6s | loss  0.5182
time    65.5s | loss  0.5312
time    67.5s | loss  0.5490
time    69.5s 

In [4]:
# --- Build a binary benchmark suite: (name, (X_train, X_test, y_train, y_test)) ---

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import (
    load_breast_cancer,
    load_iris,
    load_wine,
    load_digits,
    make_classification,
    make_moons,
    make_circles,
    make_blobs,
)

def make_benchmark_from_arrays(X, y, test_size=0.5, seed=0, scale=True):
    """
    X: (n, p) numeric
    y: (n,) binary labels {0,1} or any two values (will be converted to {0,1})
    returns: (X_train, X_test, y_train, y_test)
    """
    X = np.asarray(X)
    y = np.asarray(y)

    # map y to {0,1} if needed
    classes = np.unique(y)
    assert len(classes) == 2, f"Expected binary y; got classes={classes}"
    y01 = (y == classes[1]).astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y01, test_size=test_size, random_state=seed, stratify=y01
    )

    if scale:
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

    return (X_train, X_test, y_train.astype(int), y_test.astype(int))


def make_benchmark(loader_fn, test_size=0.5, seed=0, scale=True):
    X, y = loader_fn(return_X_y=True)
    return make_benchmark_from_arrays(X, y, test_size=test_size, seed=seed, scale=scale)


def make_binary_subset(loader_fn, class_a, class_b, test_size=0.5, seed=0, scale=True):
    """
    Take a multiclass sklearn dataset and extract a binary subset: class_a vs class_b.
    Labels are mapped to {0,1} with class_b -> 1.
    """
    X, y = loader_fn(return_X_y=True)
    mask = (y == class_a) | (y == class_b)
    X = X[mask]
    y = y[mask]
    y = (y == class_b).astype(int)
    return make_benchmark_from_arrays(X, y, test_size=test_size, seed=seed, scale=scale)


benchmarks = []

# --- Real binary dataset ---
benchmarks.append(("breast_cancer", make_benchmark(load_breast_cancer, seed=0)))

# --- Your original synthetic classification datasets ---
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.01,
    random_state=0,
)
benchmarks.append(("synthetic_cls_20f", make_benchmark_from_arrays(X, y, seed=0)))

X, y = make_classification(
    n_samples=5000,
    n_features=50,
    n_informative=15,
    n_redundant=10,
    n_clusters_per_class=2,
    class_sep=0.8,
    flip_y=0.02,
    random_state=1,
)
benchmarks.append(("synthetic_cls_50f", make_benchmark_from_arrays(X, y, seed=0)))

# --- Binary subsets from common multiclass datasets ---
benchmarks.append(("iris_0v1", make_binary_subset(load_iris, 0, 1, seed=0)))
benchmarks.append(("wine_0v1", make_binary_subset(load_wine, 0, 1, seed=0)))
benchmarks.append(("digits_0v1", make_binary_subset(load_digits, 0, 1, seed=0)))

# --- Nonlinear synthetic datasets ---
X, y = make_moons(n_samples=3000, noise=0.25, random_state=0)
benchmarks.append(("moons", make_benchmark_from_arrays(X, y, seed=0)))

X, y = make_circles(n_samples=3000, noise=0.15, factor=0.5, random_state=0)
benchmarks.append(("circles", make_benchmark_from_arrays(X, y, seed=0)))

# --- Gaussian blobs ---
X, y = make_blobs(n_samples=4000, centers=2, n_features=10, cluster_std=3.0, random_state=0)
benchmarks.append(("blobs_2c_10f", make_benchmark_from_arrays(X, y, seed=0)))

# --- Harder synthetic classification ---
X, y = make_classification(
    n_samples=4000,
    n_features=30,
    n_informative=8,
    n_redundant=10,
    n_clusters_per_class=3,
    class_sep=0.6,
    flip_y=0.05,
    random_state=2,
)
benchmarks.append(("synthetic_hard_30f", make_benchmark_from_arrays(X, y, seed=0)))

print("Total benchmarks:", len(benchmarks))
print("Benchmarks:", [name for name, _ in benchmarks])


Total benchmarks: 10
Benchmarks: ['breast_cancer', 'synthetic_cls_20f', 'synthetic_cls_50f', 'iris_0v1', 'wine_0v1', 'digits_0v1', 'moons', 'circles', 'blobs_2c_10f', 'synthetic_hard_30f']


In [5]:
classifier = NanoTabPFNClassifier(trained_model, device)

for name, dataset in benchmarks:
    scores = eval_model(classifier, [dataset])
    print(
        f"{name:18s} | "
        f"acc {scores['acc']:.4f} | "
        f"bacc {scores['balanced_acc']:.4f} | "
        f"auc {scores['roc_auc']:.4f}"
    )


breast_cancer      | acc 0.9263 | bacc 0.9163 | auc 0.9816
synthetic_cls_20f  | acc 0.7480 | bacc 0.7480 | auc 0.8355
synthetic_cls_50f  | acc 0.6676 | bacc 0.6676 | auc 0.8114
iris_0v1           | acc 1.0000 | bacc 1.0000 | auc 1.0000
wine_0v1           | acc 0.9692 | bacc 0.9690 | auc 0.9990
digits_0v1         | acc 0.9778 | bacc 0.9780 | auc 0.9998
moons              | acc 0.8680 | bacc 0.8680 | auc 0.9448
circles            | acc 0.9273 | bacc 0.9273 | auc 0.9815
blobs_2c_10f       | acc 1.0000 | bacc 1.0000 | auc 1.0000
synthetic_hard_30f | acc 0.6690 | bacc 0.6680 | auc 0.7711


In [7]:
import numpy as np

from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from model_module import NanoTabPFNClassifier


# ---- helpers ----
def metrics_binary(y_test, pred, prob_pos):
    return {
        "acc": accuracy_score(y_test, pred),
        "balanced_acc": balanced_accuracy_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, prob_pos),
    }


def eval_nanotabpfn(X_train, X_test, y_train, y_test, model, device):
    clf = NanoTabPFNClassifier(model, device)
    clf.fit(X_train, y_train)
    prob = clf.predict_proba(X_test)          # (n,2)
    pred = prob.argmax(axis=1)
    return metrics_binary(y_test, pred, prob[:, 1])


def eval_rf(X_train, X_test, y_train, y_test, seed=0):
    rf = RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=seed,
    )
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    prob = rf.predict_proba(X_test)[:, 1]
    return metrics_binary(y_test, pred, prob)


def eval_svm_rbf(X_train, X_test, y_train, y_test):
    # SVM needs scaling (you already scale in benchmark builder)
    svm = SVC(kernel="rbf", C=10.0, gamma="scale", probability=True)
    svm.fit(X_train, y_train)
    pred = svm.predict(X_test)
    prob = svm.predict_proba(X_test)[:, 1]
    return metrics_binary(y_test, pred, prob)


def eval_xgb(X_train, X_test, y_train, y_test, seed=0):
    try:
        from xgboost import XGBClassifier
    except Exception as e:
        return None, f"xgboost not available: {e}"

    xgb = XGBClassifier(
        n_estimators=1000,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=-1,
        random_state=seed,
    )
    xgb.fit(X_train, y_train)
    pred = xgb.predict(X_test).astype(int)
    prob = xgb.predict_proba(X_test)[:, 1]
    return metrics_binary(y_test, pred, prob), None


# ---- run comparison ----
print("\nComparison: nanoTabPFN vs RF vs XGBoost vs RBF-SVM (binary datasets)")
print("-" * 124)
print(f"{'dataset':18s} | {'nanoTabPFN (acc/bacc/auc)':30s} | {'RF':18s} | {'XGB':18s} | {'SVM-RBF':18s}")
print("-" * 124)

for name, (X_train, X_test, y_train, y_test) in benchmarks:
    s_nt = eval_nanotabpfn(X_train, X_test, y_train, y_test, model, device)
    s_rf = eval_rf(X_train, X_test, y_train, y_test, seed=0)
    s_svm = eval_svm_rbf(X_train, X_test, y_train, y_test)

    s_xgb, xgb_msg = eval_xgb(X_train, X_test, y_train, y_test, seed=0)

    def fmt(s):
        return f"{s['acc']:.4f}/{s['balanced_acc']:.4f}/{s['roc_auc']:.4f}"

    xgb_str = fmt(s_xgb) if s_xgb is not None else "SKIP"
    svm_str = fmt(s_svm)

    print(
        f"{name:18s} | "
        f"{fmt(s_nt):30s} | "
        f"{fmt(s_rf):18s} | "
        f"{xgb_str:18s} | "
        f"{svm_str:18s}"
    )

if 'xgb_msg' in locals() and xgb_msg:
    print("\nNote:", xgb_msg)



Comparison: nanoTabPFN vs RF vs XGBoost vs RBF-SVM (binary datasets)
----------------------------------------------------------------------------------------------------------------------------
dataset            | nanoTabPFN (acc/bacc/auc)      | RF                 | XGB                | SVM-RBF           
----------------------------------------------------------------------------------------------------------------------------
breast_cancer      | 0.9263/0.9163/0.9816           | 0.9404/0.9390/0.9847 | 0.9544/0.9541/0.9923 | 0.9684/0.9652/0.9946
synthetic_cls_20f  | 0.7480/0.7480/0.8355           | 0.9140/0.9140/0.9675 | 0.9200/0.9200/0.9729 | 0.9220/0.9220/0.9734
synthetic_cls_50f  | 0.6676/0.6676/0.8114           | 0.8704/0.8704/0.9402 | 0.9052/0.9052/0.9626 | 0.9048/0.9048/0.9640
iris_0v1           | 1.0000/1.0000/1.0000           | 1.0000/1.0000/1.0000 | 0.9800/0.9800/1.0000 | 1.0000/1.0000/1.0000
wine_0v1           | 0.9692/0.9690/0.9990           | 0.9846/0.9857/0.9962 | 0.92